## 05 — TF-IDF Optimization Experiments (Response + Numeric)

**Goal:** Improve over the strong baseline from Notebook 03 (TF-IDF(response) + numeric → Logistic Regression), while keeping:
- identical data loading (`load_splits`)
- identical evaluation protocol (`evaluate_split`, `metrics_table`)
- clean, reproducible experiment tracking

We proceed in 3 phases:
1) Reproduce the baseline (sanity check)
2) Tune TF-IDF + Logistic Regression in a controlled way (CV on train only)
3) Evaluate best configs on val/test and compare gains


#### Cell 1 — Bootstrap (same pattern as other notebooks)

In [1]:

import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


#### Cell 2 — Imports

In [2]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from src.data.load_splits import load_splits
from src.features import add_numeric_feature_columns, get_numeric_feature_cols
from src.utils import evaluate_split, metrics_table


#### Cell 3 — Load splits (locked convention)



In [3]:
train_df, val_df, test_df = load_splits(root=ROOT)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head(2)

(51605, 6) (6451, 6) (6451, 6)


,id,task,prompt,response,label,context
0,qa_3453_gt,qa,Johnny Mathis Sings included a song that start...,Blake Edwards,0,The finished product included a number from B...
1,summarization_3622_hall,summarization,Usain Bolt says Tyson Gay should have been giv...,Usain Bolt and Tyson Gay are set to compete ag...,1,NaN


#### Cell 4 — Add numeric features

In [4]:
train_df_num = add_numeric_feature_columns(train_df.copy())
val_df_num   = add_numeric_feature_columns(val_df.copy())
test_df_num  = add_numeric_feature_columns(test_df.copy())

num_cols = get_numeric_feature_cols(train_df_num)   # <-- fix from the error you hit earlier
print("Number of numeric features:", len(num_cols))
print("First 15 numeric cols:", num_cols[:15])

Number of numeric features: 10
First 15 numeric cols: ['resp_n_chars', 'resp_n_words', 'resp_n_punct', 'resp_has_multi_excl', 'resp_has_multi_q', 'resp_has_ellipsis', 'resp_n_numbers', 'resp_n_uncertainty', 'resp_punct_per_word', 'resp_numbers_per_word']


#### Cell 5 — Experiment helpers

In [5]:
def make_tfidf_lr_pipeline(
    tfidf_params: dict | None = None,
    lr_params: dict | None = None,
    *,
    response_col: str = "response",
    numeric_cols: list[str] | None = None,
    random_state: int = 42,
):
    """
    TF-IDF on response + scaled numeric → Logistic Regression
    All params are applied INSIDE the pipeline (no leakage).
    """
    tfidf_params = tfidf_params or {}
    lr_params = lr_params or {}

    if numeric_cols is None:
        raise ValueError("numeric_cols must be provided")

    tfidf = TfidfVectorizer(**tfidf_params)

    # Note: with_mean=False is required for sparse matrices
    num_scaler = StandardScaler(with_mean=False)

    preprocess = ColumnTransformer(
        transformers=[
            ("tfidf_resp", tfidf, response_col),
            ("num", num_scaler, numeric_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    clf = LogisticRegression(
        max_iter=2000,
        random_state=random_state,
        **lr_params,
    )

    return Pipeline([
        ("preprocess", preprocess),
        ("clf", clf),
    ])


def eval_all_splits(name: str, model, train_df, val_df, test_df):
    X_train, y_train = train_df, train_df["label"].values
    X_val,   y_val   = val_df,   val_df["label"].values
    X_test,  y_test  = test_df,  test_df["label"].values

    rows = []
    rows.append(evaluate_split(f"train_{name}", model, X_train, y_train))
    rows.append(evaluate_split(f"val_{name}",   model, X_val,   y_val))
    rows.append(evaluate_split(f"test_{name}",  model, X_test,  y_test))
    return metrics_table(rows)

#### Cell 6 — Phase 1: Reproduce baseline

In [6]:
baseline_tfidf = dict(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

baseline_lr = dict(
    C=1.0,
    solver="lbfgs",
    class_weight=None,
)

baseline_model = make_tfidf_lr_pipeline(
    tfidf_params=baseline_tfidf,
    lr_params=baseline_lr,
    numeric_cols=num_cols,
)

baseline_model.fit(train_df_num, train_df_num["label"].values)
baseline_metrics = eval_all_splits("tfidf+num_baseline", baseline_model, train_df_num, val_df_num, test_df_num)
baseline_metrics


train_tfidf+num_baseline metrics:
  accuracy = 0.8906
  f1       = 0.8808
  precision= 0.8928
  recall   = 0.8691
  confusion matrix:
[[25101  2504]
 [ 3142 20858]]

val_tfidf+num_baseline metrics:
  accuracy = 0.8256
  f1       = 0.8124
  precision= 0.8128
  recall   = 0.8120
  confusion matrix:
[[2890  561]
 [ 564 2436]]

test_tfidf+num_baseline metrics:
  accuracy = 0.8259
  f1       = 0.8126
  precision= 0.8136
  recall   = 0.8117
  confusion matrix:
[[2893  558]
 [ 565 2435]]


,split,accuracy,f1,precision,recall
0,train_tfidf+num_baseline,0.890592,0.880791,0.892817,0.869083
1,val_tfidf+num_baseline,0.825608,0.812406,0.812813,0.812000
2,test_tfidf+num_baseline,0.825918,0.812615,0.813565,0.811667


#### Cell 7 — Phase 2: Controlled tuning (GridSearchCV on TRAIN only)
We tune ONLY TF-IDF + LR hyperparams, using CV on train.

Validation split is kept untouched for model selection sanity check.

In [7]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_model = make_tfidf_lr_pipeline(
    tfidf_params=baseline_tfidf,
    lr_params=baseline_lr,
    numeric_cols=num_cols,
)

# --------------------
# 7A) Small WORD grid
# --------------------
word_grid_small = {
    "preprocess__tfidf_resp__analyzer": ["word"],
    "preprocess__tfidf_resp__ngram_range": [(1, 2)],   # keep fixed first
    "preprocess__tfidf_resp__min_df": [2, 5],
    "preprocess__tfidf_resp__max_df": [0.95],
    "preprocess__tfidf_resp__sublinear_tf": [True],
    "preprocess__tfidf_resp__norm": ["l2"],
    "clf__C": [0.5, 1.0, 2.0, 4.0],
    "clf__class_weight": [None, "balanced"],
}

grid_word = GridSearchCV(
    estimator=search_model,
    param_grid=word_grid_small,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_word.fit(train_df_num, train_df_num["label"].values)

print("\n[WORD] Best CV F1:", grid_word.best_score_)
print("[WORD] Best params:", grid_word.best_params_)


# ---------------------
# 7B) Small CHAR grid
# ---------------------
char_grid_small = {
    "preprocess__tfidf_resp__analyzer": ["char_wb"],
    "preprocess__tfidf_resp__ngram_range": [(3, 5), (4, 6)],
    "preprocess__tfidf_resp__min_df": [2, 5],
    "preprocess__tfidf_resp__max_df": [0.95],
    "preprocess__tfidf_resp__sublinear_tf": [True],
    "preprocess__tfidf_resp__norm": ["l2"],
    "clf__C": [0.5, 1.0, 2.0, 4.0],
    "clf__class_weight": [None, "balanced"],
}

grid_char = GridSearchCV(
    estimator=search_model,
    param_grid=char_grid_small,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_char.fit(train_df_num, train_df_num["label"].values)

print("\n[CHAR] Best CV F1:", grid_char.best_score_)
print("[CHAR] Best params:", grid_char.best_params_)


# ------------------------
# 7C) Choose the best one
# ------------------------
if grid_word.best_score_ >= grid_char.best_score_:
    best_search = grid_word
    best_tag = "best_from_word_grid"
else:
    best_search = grid_char
    best_tag = "best_from_char_grid"

best_model = best_search.best_estimator_
print(f"\nSelected: {best_tag} | CV F1 = {best_search.best_score_:.4f}")

Fitting 3 folds for each of 16 candidates, totalling 48 fits

[WORD] Best CV F1: 0.8140845891328992
[WORD] Best params: {'clf__C': 2.0, 'clf__class_weight': 'balanced', 'preprocess__tfidf_resp__analyzer': 'word', 'preprocess__tfidf_resp__max_df': 0.95, 'preprocess__tfidf_resp__min_df': 5, 'preprocess__tfidf_resp__ngram_range': (1, 2), 'preprocess__tfidf_resp__norm': 'l2', 'preprocess__tfidf_resp__sublinear_tf': True}
Fitting 3 folds for each of 32 candidates, totalling 96 fits

[CHAR] Best CV F1: 0.8038695432067217
[CHAR] Best params: {'clf__C': 2.0, 'clf__class_weight': 'balanced', 'preprocess__tfidf_resp__analyzer': 'char_wb', 'preprocess__tfidf_resp__max_df': 0.95, 'preprocess__tfidf_resp__min_df': 5, 'preprocess__tfidf_resp__ngram_range': (3, 5), 'preprocess__tfidf_resp__norm': 'l2', 'preprocess__tfidf_resp__sublinear_tf': True}

Selected: best_from_word_grid | CV F1 = 0.8141


## Conclusions — TF-IDF Optimization

- Reproduced the strong baseline from Notebook 03 (TF-IDF(response)+numeric).
- Conducted controlled hyperparameter tuning and character n-gram experiments.
- No configuration improved validation or test performance beyond the baseline.
- This suggests the classical feature-based model is close to its performance ceiling on the dataset.
- Motivates the use of task-adapted neural models in the next stage.
